In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Imports

In [2]:
! pip install -q chonkie sentence-transformers faiss-cpu jsonlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.8/233.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 85.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.2/387.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 75.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.8 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65

In [31]:
from kaggle_secrets import UserSecretsClient
import wandb

from chonkie import TokenChunker

import numpy as np              
import pandas as pd             
import matplotlib.pyplot as plt 
import seaborn as sns           
import torch                    
import torch.nn as nn         
from torch.utils.data import Dataset,DataLoader
from collections import Counter 
from string import punctuation  
import warnings                 
import string
import re

from transformers import pipeline,AutoTokenizer,AutoModel,AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, CrossEncoder 
import faiss 

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

from tqdm import tqdm
import uuid
import jsonlines

from datasets import load_dataset

In [33]:
%matplotlib inline
plt.style.use('fivethirtyeight')
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

print(f"PyTorch Version: {torch.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
np.random.seed(42)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch Version: 2.10.0+cu128
NumPy Version: 2.4.6
Pandas Version: 2.3.3
CUDA Available: True
CUDA Version: 12.8
GPU Device: Tesla T4


# W&B

In [30]:
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb_api")

In [31]:
wandb.login(key=wandb_key)
wapi=wandb.Api()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [42]:
CONFIG={
    "project_name":"23f2000391-t22026",
    "model":"BiLSTM + Attention Score",
    "tokenizer":"bert-base-uncased",
    "batch_size":16,
    "split":"Stratified-K-Fold",
    "folds_num":5,
    "embedding_dim":300,
    "hidden_dim":256,
    "num_layers":2,
    "dropout":0.3,
    "lr":2e-4,
    "weight_decay":1e-2,
    "epochs":15,
    "optimizer":"AdamW",
    "loss":"CrossEntropyLoss",
    "scheduler":"ReduceLROnPlateau"
}

In [46]:
wandb.init(
    project=CONFIG["project_name"],
    name='BiLSTM+Attention',
    config=CONFIG
)


KeyboardInterrupt: 

# Dataset

In [5]:
train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
train.drop(columns='id',inplace=True)
train.head()

,prompt,A,B,C,D,E,answer
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [7]:
test=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
test.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [8]:
test.drop(columns='id',inplace=True)
test.head()

,prompt,A,B,C,D,E
0,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


# EDA

In [14]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   prompt  2000 non-null   object
 1   A       2000 non-null   object
 2   B       2000 non-null   object
 3   C       2000 non-null   object
 4   D       2000 non-null   object
 5   E       2000 non-null   object
 6   answer  2000 non-null   object
dtypes: object(7)
memory usage: 109.5+ KB


In [15]:
train.describe()

,prompt,A,B,C,D,E,answer
count,2000,2000,2000,2000,2000,2000,2000
unique,1758,316,328,303,318,320,5
top,Choose the correct answer: What is the main se...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,B
freq,4,21,21,21,21,21,490


There might be some duplicates since only 1758 out of the 2000 prompts are unique

## Duplicates

In [9]:
print(f"Number of Duplicates: {train.duplicated().sum()}")

Number of Duplicates: 183


In [10]:
print("Shape before dropping duplicates",train.shape)
train.drop_duplicates(inplace=True)
print("Shape after dropping duplicates",train.shape)

Shape before dropping duplicates (2000, 7)
Shape after dropping duplicates (1817, 7)


## Class Distribution

In [18]:
train['answer'].value_counts()

answer
B    442
C    423
D    329
A    328
E    295
Name: count, dtype: int64

## Prompt Word Length

In [19]:
prompt_word_length=train['prompt'].apply(lambda x:len(x.split()))
prompt_word_length.describe()

count    1817.000000
mean       18.057237
std         6.844080
min         3.000000
25%        14.000000
50%        17.000000
75%        22.000000
max        51.000000
Name: prompt, dtype: float64

In [20]:
prompt_char_len=train["prompt"].str.len()
prompt_char_len.describe()

count    1817.000000
mean      117.125482
std        44.955542
min        19.000000
25%        87.000000
50%       111.000000
75%       141.000000
max       337.000000
Name: prompt, dtype: float64

## Options Word Length

In [21]:
word_lens=[]
for col in ["A","B","C","D","E"]:
    word_lens.append(train[col].str.split().str.len())

In [22]:
for i in range(len(word_lens)):
    print(f"Option {chr(i+ord("A"))}")
    print(word_lens[i].describe())

Option A
count    1817.000000
mean       26.202532
std        17.113470
min         1.000000
25%        13.000000
50%        22.000000
75%        37.000000
max        81.000000
Name: A, dtype: float64
Option B
count    1817.000000
mean       26.539351
std        18.331226
min         1.000000
25%        12.000000
50%        23.000000
75%        36.000000
max       118.000000
Name: B, dtype: float64
Option C
count    1817.000000
mean       26.652174
std        17.099736
min         1.000000
25%        15.000000
50%        25.000000
75%        36.000000
max        82.000000
Name: C, dtype: float64
Option D
count    1817.000000
mean       26.162906
std        17.158592
min         1.000000
25%        15.000000
50%        22.000000
75%        37.000000
max        78.000000
Name: D, dtype: float64
Option E
count    1817.000000
mean       26.231701
std        17.613437
min         1.000000
25%        13.000000
50%        23.000000
75%        36.000000
max       105.000000
Name: E, dtype: flo

## Correct Answer Word Length

In [23]:
correct_lengths=[]
for _, row in train.iterrows():
    correct_lengths.append(len(row[row["answer"]].split()))

pd.Series(correct_lengths).describe()

count    1817.000000
mean       28.714915
std        18.126974
min         1.000000
25%        16.000000
50%        28.000000
75%        40.000000
max       105.000000
dtype: float64

## Categories of Questions

In [13]:
'''zs=pipeline("zero-shot-classification")
zs'''

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

ZeroShotClassificationPipeline: {'model': 'BartForSequenceClassification', 'dtype': 'float32', 'device': 'cuda', 'input_modalities': 'text'}

In [14]:
'''results = zs(
    train["prompt"].tolist(),
    candidate_labels=["Physics","Chemistry","Biology","Mathematics","Computer Science","Engineering","Medicine","Astronomy","Philosophy","History","Economics","Politics","Geography","Language and Literature","Religion","General Knowledge"],
    batch_size=16,
    multi_label=False
)

cats=[result["labels"][0] for result in results]
cats[:5]'''

['Philosophy', 'Physics', 'Astronomy', 'Philosophy', 'Physics']

In [15]:
'''train["category"]=cats
train["category"].value_counts()'''

category
Physics                    506
Mathematics                350
General Knowledge          318
Astronomy                  220
Philosophy                 100
History                     74
Biology                     55
Engineering                 52
Medicine                    36
Chemistry                   28
Geography                   24
Language and Literature     20
Computer Science            20
Economics                   14
Name: count, dtype: int64

# Preprocessing

In [11]:
def clean_text(df):
    for col in test.columns:
        if col!="answer":
            df[f"clean_{col}"]=df[col].str.lower().apply(lambda x: str(x).translate(str.maketrans("", "", string.punctuation)))
            df[f"clean_{col}"]=df[f"clean_{col}"].apply(lambda x: re.sub(r"\s+", " ", x).strip())
    return df
    

In [12]:
train=clean_text(train)
train.head()

,prompt,A,B,C,D,E,answer,clean_prompt,clean_A,clean_B,clean_C,clean_D,clean_E
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
1,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is acceleratorbased lightion fusion,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...
2,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,determine the correct option what is the term ...,blueshifting,redshifting,reddening,whitening,yellowing
3,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,select the most accurate option what is martin...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
4,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,identify the correct statement what is the con...,simultaneity is relative meaning that two even...,simultaneity is relative meaning that two even...,simultaneity is absolute meaning that two even...,simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...


In [14]:
train["clean_combined_text"]=(
    "Question: "+train["clean_prompt"] +
    "\nA: "+train["clean_A"] +
    "\nB: "+train["clean_B"] +
    "\nC: "+train["clean_C"] +
    "\nD: "+train["clean_D"] +
    "\nE: "+train["clean_E"]
)

print(train["clean_combined_text"][0])

Question: pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options
A: martin heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end the relationship to the past involves acknowledging it as a historical era and the relationship to the future involves creating a world that will endure beyond ones own time
B: martin heidegger believes that humans do not exist inside time but that they are time the relationship to the past is a present awareness of having been and the relationship to the future involves anticipating a potential possibility task or engagement
C: martin heidegger does not believe in the existence of time or that it has any effect on human consciousness the relationship to the past and the future is insignificant and human existence is solely based on the present
D: martin heidegger believes that the relationship between time a

## Corpus for vocabulary

In [15]:
text=' '.join(train['clean_combined_text'].tolist())
words=text.split()
print("Total words in corpus:",len(words))
print("Total unique words in corpus:",len(set(words)))

Total words in corpus: 282930
Total unique words in corpus: 3102


In [16]:
word_freq=Counter(words)
print(f"Most Frequent Words:")
for i,j in (word_freq.most_common(10)):
    print(f"{i}: {j}")
print(f"\nLeast Frequent Words:")
for i,j in (word_freq.most_common()[-11:-1][::-1]):
    print(f"{i}: {j}")

Most Frequent Words:
the: 24709
of: 13223
a: 10729
is: 9327
and: 6326
in: 6131
to: 5771
that: 4037
by: 2015
an: 1843

Least Frequent Words:
subsystems: 1
increases: 1
subsystem: 2
ecosystem: 2
decreases: 2
subframework: 2
lowers: 2
submechanisms: 2
differences: 2
boosts: 3


## Split

In [17]:
#train_df,val_df=train_test_split(train,test_size=0.2,stratify=train["answer"],random_state=42)

skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

#print(train_df["answer"].value_counts()/len(train_df))
#print(val_df["answer"].value_counts()/len(val_df))

# Scoring

In [18]:
def map_at_3(true,pred):
    scores=[]
    for actual,preds in zip(true,pred):
        score=0.0
        for rank,pred in enumerate(preds,start=1):
            if pred==actual:
                score=1.0/rank
                break
        scores.append(score)
    return np.mean(scores)

In [19]:
def top3_accuracy(y_true, predictions):
    return np.mean([truth in pred for truth, pred in zip(y_true, predictions)])

In [20]:
def top1_accuracy(y_true, predictions):
    top1=[pred[0] if len(pred) else "Z" for pred in predictions]

    return accuracy_score(y_true,top1)

# PreTrained Models

## Zero-Shot Classification

In [ ]:
'''qwen_model_name="Qwen/Qwen2.5-7B-Instruct"
qwen_tokenizer=AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model=AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    dtype=torch.float16,
    device_map="auto"
)'''

In [ ]:
'''def create_zero_shot_prompt(row):
    zero_shot_prompt=f"""
    You are solving a multiple choice question containing 5 choices.
    Question:
    {row["prompt"]}
    Choices:
    A. {row["A"]}
    B. {row["B"]}
    C. {row["C"]}
    D. {row["D"]}
    E. {row["E"]}
    Return ONLY the three most likely answer labels with a single space separating them.
    Example:
    C A D
    """
    return zero_shot_prompt'''

In [ ]:
'''def create_few_shot_prompt(row):
    few_shot_examples = []
    for label in ["A", "B", "C", "D", "E"]:
        example = train_df[train_df["answer"] == label].sample(n=1,random_state=42).iloc[0]
        few_shot_examples.append(example)
    prompt = """You are an expert at solving multiple-choice questions.
                Below are some solved examples.\n"""

    for i, ex in enumerate(few_shot_examples, 1):

        prompt += f"""Example {i}
        Question: {ex["prompt"]}
        
        Choices:
        A. {ex["A"]}
        B. {ex["B"]}
        C. {ex["C"]}
        D. {ex["D"]}
        E. {ex["E"]}
        
        Correct Answer:
        {ex["answer"]}
        
        """
        
    prompt += f"""
    Now answer the following question.
    
    Question:
    {row["prompt"]}
    
    Choices:
    A. {row["A"]}
    B. {row["B"]}
    C. {row["C"]}
    D. {row["D"]}
    E. {row["E"]}
    
    Rank the choices based on their probability of being the correct answer.
    Then return the top three answer labels separated by spaces.
    
    Example output:
    C A D
    
    Do not explain your answer.
    """

    return prompt'''

In [ ]:
'''def predict_qwen(row):

    prompt = create_few_shot_prompt(row)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    ).to(qwen_model.device)

    with torch.no_grad():

        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id
        )

    generated = outputs[0][inputs.input_ids.shape[1]:]

    prediction = qwen_tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    labels = re.findall(r"\b[A-E]\b", prediction)

    return labels[:3]'''

In [ ]:
'''predictions = []

fold_map = []
fold_acc = []
fold_top3 = []

for fold,(train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):
    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)
    predictions=[]
    for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
        pred = predict_qwen(row)
        predictions.append(pred)
        torch.cuda.empty_cache()
    map3 = map_at_3(
        val_df["answer"],
        predictions
    )
    
    acc = top1_accuracy(
        val_df["answer"],
        predictions
    )
    
    top3 = top3_accuracy(
        val_df["answer"],
        predictions
    )
    wandb.log({
        "Fold": fold + 1,
        "MAP@3": map3,
        "Accuracy": acc,
        "Top3 Accuracy": top3
    })
    fold_map.append(map3)
    fold_acc.append(acc)
    fold_top3.append(top3)
print(f"MAP@3 : {np.mean(fold_map):.4f}")
print(f"Accuracy : {np.mean(fold_acc):.4f}")
print(f"Top3 Accuracy : {np.mean(fold_top3):.4f}")'''

In [ ]:
'''wandb.log({
    "Average MAP@3": np.mean(fold_map),
    "Average Accuracy": np.mean(fold_acc),
    "Average Top3 Accuracy": np.mean(fold_top3)
})

wandb.finish()'''

# Model from Scratch

## BiLSTM + Attention Scores

In [25]:
tokenizer=AutoTokenizer.from_pretrained(CONFIG['tokenizer'])
tokenizer

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [50]:
class Attention(nn.Module):
    def __init__(self,hidden_dim):
        super().__init__()
        self.linear=nn.Linear(hidden_dim*2,1)

    def forward(self,x):
        weights=torch.softmax(self.linear(x),dim=1)
        context=(weights*x).sum(dim=1)
        return context

In [51]:
class BiLSTMAttention(nn.Module):
    def __init__(self,vocab_size,embedding_dim=300,hidden_dim=256,num_layers=2,dropout=0.3):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embedding_dim,padding_idx=0)
        self.lstm=nn.LSTM(embedding_dim,hidden_dim,num_layers=num_layers,batch_first=True,bidirectional=True,dropout=dropout)
        self.attention=Attention(hidden_dim)
        self.dropout=nn.Dropout(dropout)
        self.fc=nn.Linear(hidden_dim*2,1)

    def forward(self,input_ids):
        batch_size=input_ids.size(0)
        scores=[]
        for i in range(5):
            x=input_ids[:,i]
            x=self.embedding(x)
            output,_=self.lstm(x)
            context=self.attention(output)
            context=self.dropout(context)
            score=self.fc(context)
            scores.append(score.squeeze(1))
        scores=torch.stack(scores,dim=1)
        return scores

In [52]:
def encode_question(question,option):
    encoding=tokenizer(
        question,
        option,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_attention_mask=False,
        return_token_type_ids=False
    )

    return encoding["input_ids"]

In [53]:
label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

id2label = {
    0:"A",
    1:"B",
    2:"C",
    3:"D",
    4:"E"
}

In [54]:
class MCQDataset(Dataset):
    def __init__(self, dataframe):
        self.df=dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        inputs=[]
        for choice in ["A","B","C","D","E"]:
            ids=encode_question(row["prompt"],row[choice])
            inputs.append(ids)

        inputs=torch.tensor(inputs)
        label=label2id[row["answer"]]
        return {"input_ids": inputs,"label": torch.tensor(label)}

In [55]:
def train_one_epoch(model,dataloader,optimizer,criterion,device):
    model.train()
    running_loss=0

    for batch in tqdm(dataloader):
        input_ids=batch["input_ids"].to(device)
        labels=batch["label"].to(device)
        optimizer.zero_grad()
        outputs=model(input_ids)
        loss=criterion(outputs,labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)

        optimizer.step()
        running_loss+=loss.item()

    epoch_loss=running_loss/len(dataloader)

    return epoch_loss

In [56]:
def validate(model,dataloader,criterion,device):
    model.eval()
    running_loss=0
    predictions=[]
    ground_truth=[]
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids=batch["input_ids"].to(device)
            labels=batch["label"].to(device)

            outputs=model(input_ids)
            loss=criterion(outputs, labels)
            running_loss+=loss.item()

            probs=torch.softmax(outputs, dim=1)

            top3=torch.topk(probs,k=3,dim=1).indices.cpu().numpy()

            for pred in top3:
                predictions.append([id2label[i] for i in pred])

            ground_truth.extend([id2label[i.item()] for i in labels])

    val_loss=running_loss/len(dataloader)
    map3=map_at_3(ground_truth,predictions)
    acc=top1_accuracy(ground_truth,predictions)
    top3_acc=top3_accuracy(ground_truth,predictions)

    return (val_loss,map3,acc,top3_acc,predictions)

In [58]:
fold_map3=[]
fold_top1=[]
fold_top3=[]
best_cv_map3=-1
for fold,(train_idx,val_idx) in enumerate(skf.split(train,train["answer"])):

    print(f"Fold {fold+1}")

    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)

    train_dataset=MCQDataset(train_df)
    val_dataset=MCQDataset(val_df)
    
    train_loader=DataLoader(train_dataset,batch_size=CONFIG["batch_size"],shuffle=True)
    val_loader=DataLoader(val_dataset,batch_size=CONFIG["batch_size"],shuffle=False)

    bilstm_model=BiLSTMAttention(
        vocab_size=tokenizer.vocab_size,
        embedding_dim=300,
        hidden_dim=256,
        num_layers=2,
        dropout=0.3
    )
    bilstm_model.to(device)
    
    criterion=nn.CrossEntropyLoss()
    optimizer=torch.optim.AdamW(bilstm_model.parameters(),lr=2e-4,weight_decay=1e-2)

    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="max",factor=0.5,patience=2)
    
    best_map3=-1
    best_top3=0
    best_top1=0
    patience=3
    counter=0

    for epoch in range(CONFIG["epochs"]):

        print(f"\nEpoch {epoch+1}/{CONFIG["epochs"]}")
        train_loss=train_one_epoch(bilstm_model,train_loader,optimizer,criterion,device)
        (val_loss,map3,acc,top3_acc,predictions)=validate(bilstm_model,val_loader,criterion,device)
    
        print(f"Train Loss:{train_loss:.4f}")
        print(f"Val Loss  :{val_loss:.4f}")
        print(f"MAP@3     :{map3:.4f}")
        print(f"Top1 Acc  :{acc:.4f}")
        print(f"Top3 Acc  :{top3_acc:.4f}")

        scheduler.step(map3)

        wandb.log({
            "Fold":fold+1,
            "Epoch":epoch+1,
            "Train Loss":train_loss,
            "Validation Loss":val_loss,
            "Validation MAP@3":map3,
            "Validation Top1 Accuracy":acc,
            "Validation Top3 Accuracy":top3_acc,
            "Learning Rate":optimizer.param_groups[0]["lr"]
            })

        if map3>best_map3:
            best_map3=map3
            best_top1=acc
            best_top3=top3_acc
            counter=0

            if best_map3>best_cv_map3:
                best_cv_map3=map3
                torch.save(bilstm_model.state_dict(),"bilstm_best_model.pt")
        else:
            counter += 1
            if counter >= patience:
                print("Early Stopping")
                break
    fold_map3.append(best_map3)
    fold_top1.append(best_top1)
    fold_top3.append(best_top3)
    
    wandb.log({
        "Fold":fold+1,
        "Fold MAP@3":best_map3,
        "Fold Top1":best_top1,
        "Fold Top3":best_top3
        })

Fold 1

Epoch 1/15


100%|██████████| 23/23 [00:01<00:00, 18.03it/s]


Train Loss:1.4435
Val Loss  :0.9987
MAP@3     :0.7624
Top1 Acc  :0.6429
Top3 Acc  :0.9148

Epoch 2/15


100%|██████████| 23/23 [00:01<00:00, 17.78it/s]


Train Loss:0.6561
Val Loss  :0.4128
MAP@3     :0.9272
Top1 Acc  :0.8764
Top3 Acc  :0.9780

Epoch 3/15


...

Train Loss:0.0242
Val Loss  :0.0179
MAP@3     :1.0000
Top1 Acc  :1.0000
Top3 Acc  :1.0000

Epoch 7/15


100%|██████████| 23/23 [00:01<00:00, 17.97it/s]


Train Loss:0.0122
Val Loss  :0.0103
MAP@3     :0.9986
Top1 Acc  :0.9972
Top3 Acc  :1.0000

Epoch 8/15


100%|██████████| 23/23 [00:01<00:00, 18.00it/s]

Train Loss:0.0028
Val Loss  :0.0005
MAP@3     :1.0000
Top1 Acc  :1.0000
Top3 Acc  :1.0000
Early Stopping


In [ ]:
print(f"MAP@3: {np.mean(fold_map3):.4f}}")
print(f"Top1 : {np.mean(fold_top1):.4f}}")
print(f"Top3 : {np.mean(fold_top3):.4f}}")

wandb.log({
    "MAP@3 Mean": np.mean(fold_map3),
    "Top1 Mean": np.mean(fold_top1),
    "Top3 Mean": np.mean(fold_top3),
})

In [ ]:
artifact=wandb.Artifact("bilstm_best_model",type="model")
artifact.add_file("/kaggle/working/bilstm_best_model.pt")
wandb.log_artifact(artifact)
wandb.finish()

In [59]:
class TestDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        inputs = []
        for choice in ["A","B","C","D","E"]:
            ids = encode_question(row["prompt"],row[choice])
            inputs.append(ids)

        return {"input_ids": torch.tensor(inputs)}

In [60]:
test_dataset=TestDataset(test)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

In [61]:
bilstm_model = BiLSTMAttention(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=300,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3
).to(device)

bilstm_model.load_state_dict(
    torch.load(
        "bilstm_best_model.pt",
        map_location=device
    )
)

bilstm_model.eval()

BiLSTMAttention(
  (embedding): Embedding(30522, 300, padding_idx=0)
  (lstm): LSTM(300, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (attention): Attention(
    (linear): Linear(in_features=512, out_features=1, bias=True)
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=1, bias=True)
)

In [62]:
def bilstm_predict(model,dataloader,device):
    model.eval()
    predictions=[]
    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids=batch["input_ids"].to(device)
            outputs=model(input_ids)
            probs=torch.softmax(outputs,dim=1)

            top3=torch.topk(probs,k=3,dim=1).indices.cpu().numpy()

            for pred in top3:
                letters=[id2label[i] for i in pred]
                predictions.append(" ".join(letters))

    return predictions

In [64]:
test_predictions=bilstm_predict(bilstm_model,test_loader,device)
test_predictions[:5]

100%|██████████| 16/16 [00:01<00:00, 10.08it/s]


['A E B', 'B E C', 'B E D', 'E C A', 'C A D']

# Model of Choice

## Tf-Idf + Logistic Regression

In [ ]:
'''def build_pairwise_dataset(df,test=False):
    rows=[]
    options=["A","B","C","D","E"]
    for qid,row in df.iterrows():
        for c in options:
            if not test:
                rows.append({
                    "text":"Question: "+row["clean_prompt"]+"\nAnswer: "+row[f"clean_{c}"],
                    "label":1 if c==row["answer"] else 0,
                    "question_id":qid,
                    "option":c
                })
            else:
                rows.append({
                    "text":"Question: "+row["clean_prompt"]+"\nAnswer: "+row[f"clean_{c}"],
                    "question_id":qid,
                    "option":c
                })

    return pd.DataFrame(rows)'''

In [ ]:
'''vectorizer=TfidfVectorizer(
    ngram_range=(1,2),
    stop_words="english",
    min_df=3,
    sublinear_tf=True
)
clf=LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42)'''

In [ ]:
'''fold_map3=[]
fold_acc=[]
fold_top3=[]

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):
    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)
    
    print("Fold:",fold+1)
    
    pair_train=build_pairwise_dataset(train_df)
    pair_val=build_pairwise_dataset(val_df)

    X_train=vectorizer.fit_transform(pair_train["text"])
    X_val=vectorizer.transform(pair_val["text"])

    clf.fit(X_train,pair_train["label"])
    
    probs=clf.predict_proba(X_val)[:,1]

    predictions=[]
    choices=["A","B","C","D","E"]
    for i in range(len(val_df)):
        start=i*5
        end=start+5
        scores=probs[start:end]
        ranked=np.argsort(scores)[::-1]
        top3=[choices[j] for j in ranked[:3]]
        predictions.append(top3)
       
    map3=map_at_3(val_df["answer"],predictions)
    acc=top1_accuracy(val_df["answer"],predictions)
    top3=top3_accuracy(val_df["answer"],predictions)

    print(f"MAP@3: {map3:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Top3 Accuracy: {top3:.4f}")

    fold_map3.append(map3)
    fold_acc.append(acc)
    fold_top3.append(top3)

    wandb.log({
        "Fold": fold + 1,
        "MAP@3": map3,
        "Top1 Accuracy": acc,
        "Top3 Accuracy": top3
    })'''

In [ ]:
'''print("Average MAP@3:", np.mean(fold_map3))
print("Average Accuracy:", np.mean(fold_acc))
print("Average Top3 Accuracy:", np.mean(fold_top3))

wandb.log({
    "Average MAP@3": np.mean(fold_map3),
    "Average Top1 Accuracy": np.mean(fold_acc),
    "Average Top3 Accuracy": np.mean(fold_top3)
})

wandb.finish()'''

In [ ]:
'''pair_train=build_pairwise_dataset(train)
X_train=vectorizer.fit_transform(pair_train["text"])
clf.fit(X_train,pair_train["label"])'''

In [ ]:
'''test=clean_text(test)
test.head()'''

In [ ]:
'''pair_test=build_pairwise_dataset(test,True)
X_test=vectorizer.transform(pair_test["text"])
probs=clf.predict_proba(X_test)[:, 1]
probs'''

In [ ]:
'''choices=["A","B","C","D","E"]
test_predictions=[]
for i in range(len(test)):
    scores=probs[i*5:(i+1)*5]
    ranked=np.argsort(scores)[::-1]
    top3=[choices[j] for j in ranked[:3]]
    test_predictions.append(" ".join(top3))
test_predictions[:5]'''

## RAG System

In [ ]:
'''wiki=load_dataset("wikimedia/wikipedia","20231101.en",split="train")
wiki=wiki.select(range(50000))'''

In [ ]:
'''def clean_data(text):
    text=text.lower()
    text=re.sub(r"\s+"," ",text)
    text=re.sub(r"\[[^\]]*\]","",text)
    text=text.strip()
    return text'''

In [ ]:
'''chunker=TokenChunker(chunk_size=CONFIG["chunk_size"],chunk_overlap=CONFIG["chunk_overlap"])
chunker'''

In [25]:
'''model = SentenceTransformer("BAAI/bge-base-en-v1.5") 
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
llm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct",dtype=torch.float32, device_map="auto")'''

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
'''kb=[]
chunk_records=[]
for article in wiki:
    title=article["title"]
    text=clean_data(article["text"])
    url=article.get("url", "")

    if len(text.strip()) == 0:
        continue

    chunks=chunker(text)

    for chunk_idx, chunk in enumerate(chunks):
        chunk_text=chunk.text
        kb.append(chunk_text)
        chunk_records.append({
            "chunk_id":str(uuid.uuid4()),
            "chunk_index": chunk_idx,
            "title": title,
            "url": url,
            "text": chunk_text,
            "token_count": chunk.token_count
        })

print(f"Total chunks generated: {len(chunk_records)}")'''

In [ ]:
'''with jsonlines.open('/kaggle/working/knowledge_base.jsonl', mode='w') as writer:
    writer.write_all(chunk_records)

wandb.config.update({"total_chunks": len(chunk_records)})

print(f"Chunk and metadata saved")'''

In [ ]:
'''texts_to_embed=[chunk["text"] for chunk in chunk_records]
embeddings = model.encode(
    texts_to_embed,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)'''

In [ ]:
'''index_path="/kaggle/working/vector_index.idx"

embedding_dimension=embeddings.shape[1]
faiss_index=faiss.IndexFlatL2(embedding_dimension) 
faiss_index.add(embeddings)

faiss.write_index(faiss_index, index_path)

wandb.config.update({"embedding_dimension": embedding_dimension})
print(f"Saved FAISS index to {index_path}")'''

In [ ]:
'''print("Logging data and index to Weights & Biases...")

artifact = wandb.Artifact(
    name="wikipedia_knowledge_base",
    type="dataset",
    description="JSONL chunks and FAISS index for Wikipedia Dataset",
    metadata={
        "chunking_strategy": "TokenChunker",
        "chunk_size_tokens": CONFIG["chunk_size"],
        "chunk_overlap_tokens": CONFIG["chunk_overlap"],
        "total_chunks_generated": len(chunk_records),
        
        "embedding_model": CONFIG["embedding_model"],
        "embedding_dimension": embedding_dimension,
        "vector_index_type": "FAISS IndexFlatL2",
        
        "source_material_type": "Wikipedia",
    }
)

artifact.add_file("/kaggle/working/knowledge_base.jsonl")
artifact.add_file("/kaggle/working/vector_index.idx")
wandb.log_artifact(artifact)

print("Chunking and Indexing complete and pushed to W&B")'''

In [32]:
'''wapi.artifact("23f2000391-dl-genai-project/23f2000391-t22026/wikipedia_knowledge_base:v0").download()'''

wandb: Downloading large artifact 'wikipedia_knowledge_base:v0', 3965.68MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:42.1 (94.3MB/s)


'/kaggle/working/artifacts/wikipedia_knowledge_base:v0'

In [33]:
'''jsonl_path="/kaggle/working/artifacts/wikipedia_knowledge_base:v0/knowledge_base.jsonl"
index_path="/kaggle/working/artifacts/wikipedia_knowledge_base:v0/vector_index.idx"'''

In [34]:
'''print("Loading Knowledge Base")
chunks = []
with jsonlines.open(jsonl_path) as reader:
    for obj in reader:
        chunks.append(obj)

print("Loading FAISS Index...")
faiss_index = faiss.read_index(index_path)'''

Loading Knowledge Base
Loading FAISS Index...


In [45]:
'''def rag_predict(row,kb,index):    
    row_embeddings=model.encode(row["prompt"],convert_to_numpy=True,normalize_embeddings=True).reshape(1,-1)
    distances,indices=index.search(row_embeddings,k=20)

    results=[]
    for rank,(dist,idx) in enumerate(zip(distances[0],indices[0])):
        chunk_data=kb[idx]
        results.append({
            "rank": rank + 1,
            "distance": dist,
            "title": chunk_data["title"],
            "text": chunk_data["text"]
        })
            
    pairs=[[row["prompt"],chunk['text']] for chunk in results] 
    ce_scores=cross_encoder.predict(pairs,batch_size=16) 

    for i,score in enumerate(ce_scores):
        results[i]["rerank_score"]=score

    reranked_chunks=sorted(results,key=lambda x: x["rerank_score"],reverse=True)
    top_3_chunks=reranked_chunks[:3]

    context_text="\n\n".join([f"Context {idx+1}\nTitle: {c['title']}\nContent: {c['text']}" for idx,c in enumerate(top_3_chunks)])

    query_row=f"""
    Retrieved Context
    {context_text}
    
    Question:
    {row['prompt']}
    
    A. {row['A']}
    B. {row['B']}
    C. {row['C']}
    D. {row['D']}
    E. {row['E']}
    
    Return only the three most likely answer letters separated by spaces.
    """
    
    system_prompt="""
    You are an expert multiple-choice reasoning assistant.
    Use the retrieved context to help answer the question.
    If the context is insufficient, rely on your own reasoning.
    Return ONLY the top three answer letters separated by spaces.
    
    Example:
    A C D
    
    Do not explain your reasoning.
    """
    messages=[
        {"role": "system","content": system_prompt},
        {"role": "user","content": query_row}
    ]

    text_prompt=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=tokenizer(text_prompt,return_tensors="pt",truncation=True,max_length=4096).to(llm.device)
    
    with torch.no_grad():
        with torch.autocast("cuda"):
            outputs = llm.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
    
    response=tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    labels=re.findall(r"\b[A-E]\b", response.upper())
    labels=list(dict.fromkeys(labels))
    
    for letter in "ABCDE":
        if letter not in labels:
            labels.append(letter)
    
    return labels[:3]
'''

In [46]:
'''fold_map3=[]
fold_acc=[]
fold_top3=[]
for fold, (train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):
    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)
    
    print("Fold:",fold+1)
    
    predictions=[]
    for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
        top3_preds=rag_predict(row,chunks,faiss_index)        
        predictions.append(top3_preds)
       
    map3=map_at_3(val_df["answer"],predictions)
    acc=top1_accuracy(val_df["answer"],predictions)
    top3=top3_accuracy(val_df["answer"],predictions)

    print(f"MAP@3: {map3:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Top3 Accuracy: {top3:.4f}")

    fold_map3.append(map3)
    fold_acc.append(acc)
    fold_top3.append(top3)

    wandb.log({
        "Fold": fold + 1,
        "MAP@3": map3,
        "Top1 Accuracy": acc,
        "Top3 Accuracy": top3
    })
'''

Fold: 1


100%|██████████| 364/364 [06:28<00:00,  1.07s/it]


MAP@3: 0.7587
Accuracy: 0.6511
Top3 Accuracy: 0.8874
Fold: 2


100%|██████████| 364/364 [06:12<00:00,  1.02s/it]


MAP@3: 0.7601
Accuracy: 0.6484
Top3 Accuracy: 0.8984
Fold: 3


...

MAP@3: 0.7328
Accuracy: 0.6143
Top3 Accuracy: 0.8898
Fold: 4


100%|██████████| 363/363 [06:09<00:00,  1.02s/it]


MAP@3: 0.7879
Accuracy: 0.6887
Top3 Accuracy: 0.9146
Fold: 5


100%|██████████| 363/363 [06:09<00:00,  1.02s/it]

MAP@3: 0.7484
Accuracy: 0.6529
Top3 Accuracy: 0.8650


In [47]:
'''print("Average MAP@3:", np.mean(fold_map3))
print("Average Accuracy:", np.mean(fold_acc))
print("Average Top3 Accuracy:", np.mean(fold_top3))

wandb.log({
    "Average MAP@3": np.mean(fold_map3),
    "Average Top1 Accuracy": np.mean(fold_acc),
    "Average Top3 Accuracy": np.mean(fold_top3)
})

wandb.finish()'''

Average MAP@3: 0.7575654143835961
Average Accuracy: 0.6510746828928647
Average Top3 Accuracy: 0.8910271546635181


'wandb.log({\n    "Average MAP@3": np.mean(fold_map3),\n    "Average Top1 Accuracy": np.mean(fold_acc),\n    "Average Top3 Accuracy": np.mean(fold_top3)\n})\n\nwandb.finish()'

In [48]:
'''test_predictions=[]
for _, row in tqdm(test.iterrows(), total=len(test)):
    test_predictions.append(rag_predict(row,chunks,faiss_index))
test_predictions[:5]'''

100%|██████████| 500/500 [08:33<00:00,  1.03s/it]


[['A', 'E', 'D'],
 ['B', 'C', 'A'],
 ['B', 'E', 'C'],
 ['A', 'C', 'E'],
 ['C', 'D', 'A']]

# Submission

In [65]:
submission_preds = [" ".join(pred) for pred in test_predictions]
sub=pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
sub.set_index('ID',inplace=True)
sub.head()

,Prediction
ID,
1,A B C
2,A B C
3,A B C
4,A B C
5,A B C


In [66]:
sub['Prediction']=submission_preds
sub.head()

,Prediction
ID,
1,A E B
2,B E C
3,B E D
4,E C A
5,C A D


In [41]:
sub.to_csv("submission.csv")
print("Submission File Created")

Submission File Created
